GenAI decleration: Generative AI tools from OpenAI, Anthropic, and Google have been used to help make the code in this file.

In [4]:
import gurobipy as gp
from gurobipy import GRB
import SystemCharacteristics as data
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

Professor model

In [5]:
import pyomo.environ as pyo
import numpy as np

def professor_model(price_arr, occ_r1_arr, occ_r2_arr, data):
    model = pyo.ConcreteModel()
    num_t = data['num_timeslots']
    model.T = pyo.RangeSet(0, num_t - 1)
    model.R = pyo.Set(initialize=['r1', 'r2'])

    # Mapping occupancy arrays to a dictionary for easy indexing (r, t)
    occ_data = {}
    for t in range(num_t):
        occ_data[('r1', t)] = occ_r1_arr[t]
        occ_data[('r2', t)] = occ_r2_arr[t]

    # --- Variables ---
    model.p = pyo.Var(model.R, model.T, domain=pyo.NonNegativeReals) # Heating power [cite: 7]
    model.T_in = pyo.Var(model.R, model.T, domain=pyo.Reals)         # Indoor temp [cite: 8]
    model.H = pyo.Var(model.T, domain=pyo.NonNegativeReals)          # Humidity [cite: 9]
    model.v = pyo.Var(model.T, domain=pyo.Binary)                    # Ventilation active [cite: 10]
    model.s = pyo.Var(model.T, domain=pyo.Binary)                    # Startup [cite: 11]
    model.u = pyo.Var(model.R, model.T, domain=pyo.Binary)          # Overrule active [cite: 13]
    
    # Auxiliary variables [cite: 12, 46]
    model.y_low = pyo.Var(model.R, model.T, domain=pyo.Binary) 
    model.y_ok = pyo.Var(model.R, model.T, domain=pyo.Binary)
    model.y_high = pyo.Var(model.R, model.T, domain=pyo.Binary)

    M_temp = 100.0 # Big-M [cite: 25]
    M_hum = 100.0

    # --- Objective Function ---
    # Minimize total cost using the provided price array [cite: 31, 32, 33]
    def obj_rule(m):
        return sum(price_arr[t] * (data['ventilation_power'] * m.v[t] + 
                                   sum(m.p[r, t] for r in m.R)) for t in m.T)
    model.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

    # --- Constraints ---
    for r in model.R:
        model.T_in[r, 0].fix(data['initial_temperature'])
    model.H[0].fix(data['initial_humidity'])

    # Temperature Dynamics with dynamic occupancy [cite: 35, 36]
    def temp_dynamics_rule(m, r, t):
        if t == 0: return pyo.Constraint.Skip
        r_other = 'r2' if r == 'r1' else 'r1'
        return m.T_in[r, t] == (m.T_in[r, t-1] + 
                                data['heat_exchange_coeff'] * (m.T_in[r_other, t-1] - m.T_in[r, t-1]) - 
                                data['thermal_loss_coeff'] * (m.T_in[r, t-1] - data['outdoor_temperature'][t-1]) + 
                                data['heating_efficiency_coeff'] * m.p[r, t-1] - 
                                data['heat_vent_coeff'] * m.v[t-1] + 
                                data['heat_occupancy_coeff'] * occ_data[(r, t-1)])
    model.temp_dyn = pyo.Constraint(model.R, model.T, rule=temp_dynamics_rule)

    # Humidity Dynamics with dynamic occupancy [cite: 39, 40]
    def hum_dynamics_rule(m, t):
        if t == 0: return pyo.Constraint.Skip
        total_occ = sum(occ_data[(r, t-1)] for r in m.R)
        return m.H[t] == m.H[t-1] + data['humidity_occupancy_coeff'] * total_occ - data['humidity_vent_coeff'] * m.v[t-1]
    model.hum_dyn = pyo.Constraint(model.T, rule=hum_dynamics_rule)

    # High Temp Logic / Heater Deactivation [cite: 53, 54, 55]
    def high_temp_1(m, r, t):
        return m.T_in[r, t] >= data['temp_max_comfort_threshold'] - M_temp * (1 - m.y_high[r, t])
    def high_temp_2(m, r, t):
        return m.T_in[r, t] <= data['temp_max_comfort_threshold'] + M_temp * m.y_high[r, t]
    def heater_cutoff(m, r, t):
        return m.p[r, t] <= data['heating_max_power'] * (1 - m.y_high[r, t])
    model.hi_1 = pyo.Constraint(model.R, model.T, rule=high_temp_1)
    model.hi_2 = pyo.Constraint(model.R, model.T, rule=high_temp_2)
    model.hi_cut = pyo.Constraint(model.R, model.T, rule=heater_cutoff)

    # Low/OK Temp detection [cite: 58, 60, 64, 66]
    def low_temp_1(m, r, t):
        return m.T_in[r, t] <= data['temp_min_comfort_threshold'] + M_temp * (1 - m.y_low[r, t])
    def low_temp_2(m, r, t):
        return m.T_in[r, t] >= data['temp_min_comfort_threshold'] - M_temp * m.y_low[r, t]
    model.lo_1 = pyo.Constraint(model.R, model.T, rule=low_temp_1)
    model.lo_2 = pyo.Constraint(model.R, model.T, rule=low_temp_2)

    def ok_temp_1(m, r, t):
        return m.T_in[r, t] >= data['temp_OK_threshold'] - M_temp * (1 - m.y_ok[r, t])
    def ok_temp_2(m, r, t):
        return m.T_in[r, t] <= data['temp_OK_threshold'] + M_temp * m.y_ok[r, t]
    model.ok_1 = pyo.Constraint(model.R, model.T, rule=ok_temp_1)
    model.ok_2 = pyo.Constraint(model.R, model.T, rule=ok_temp_2)

    # Overrule Logic [cite: 72, 73, 75, 77, 78]
    def ovr_1(m, r, t): return m.u[r, t] >= m.y_low[r, t]
    def ovr_2(m, r, t):
        if t == 0: return pyo.Constraint.Skip
        return m.u[r, t] <= m.u[r, t-1] + m.y_low[r, t]
    def ovr_max(m, r, t): return m.p[r, t] >= data['heating_max_power'] * m.u[r, t]
    def ovr_de1(m, r, t):
        if t == 0: return pyo.Constraint.Skip
        return m.u[r, t] >= m.u[r, t-1] - m.y_ok[r, t]
    def ovr_de2(m, r, t): return m.u[r, t] <= 1 - m.y_ok[r, t]
    model.ov1 = pyo.Constraint(model.R, model.T, rule=ovr_1)
    model.ov2 = pyo.Constraint(model.R, model.T, rule=ovr_2)
    model.ovmax = pyo.Constraint(model.R, model.T, rule=ovr_max)
    model.ovde1 = pyo.Constraint(model.R, model.T, rule=ovr_de1)
    model.ovde2 = pyo.Constraint(model.R, model.T, rule=ovr_de2)

    # Ventilation Startup & Min Up-time [cite: 80, 83, 84]
    def vent_start(m, t):
        if t == 0: return m.s[t] >= m.v[t]
        return m.s[t] >= m.v[t] - m.v[t-1]
    model.v_start = pyo.Constraint(model.T, rule=vent_start)

    def vent_uptime(m, t):
        U = data['vent_min_up_time'] # U_vent [cite: 27]
        horizon = num_t # L [cite: 17]
        end_t = min(t + U, horizon)
        return sum(m.v[tau] for tau in range(t, end_t)) >= (min(U, horizon - t)) * m.s[t]
    model.v_up = pyo.Constraint(model.T, rule=vent_uptime)

    # Humidity Trigger [cite: 85, 86]
    def hum_trig(m, t):
        return m.H[t] <= data['humidity_threshold'] + M_hum * m.v[t]
    model.h_trig = pyo.Constraint(model.T, rule=hum_trig)

    # Solve
    solver = pyo.SolverFactory('gurobi')
    solver.solve(model)

    # Final result structure as requested
    HVAC_results = {
        "Temp_r1": [pyo.value(model.T_in['r1', t]) for t in model.T],
        "Temp_r2": [pyo.value(model.T_in['r2', t]) for t in model.T],
        "h_r1": [pyo.value(model.p['r1', t]) for t in model.T],
        "h_r2": [pyo.value(model.p['r2', t]) for t in model.T],
        "v": [pyo.value(model.v[t]) for t in model.T],
        "s": [pyo.value(model.s[t]) for t in model.T],
        "z_high": [pyo.value(model.y_high['r1', t]) for t in model.T],
        "z_low": [pyo.value(model.y_low['r1', t]) for t in model.T],
        "Hum": [pyo.value(model.H[t]) for t in model.T],
        "price": price_arr,
        "Occ_r1": occ_r1_arr,
        "Occ_r2": occ_r2_arr,
        "outdoor_temperature": data['outdoor_temperature'],
        "avg_cost": pyo.value(model.obj)
    }
    
    return HVAC_results

This is our adjusted plotting function (Updated to support optional axes parameter)

In [17]:
def plot_HVAC_results(HVAC_results, axes=None):
    
    Temp_r1 = HVAC_results['Temp_r1']
    Temp_r2 = HVAC_results['Temp_r2']
    h_r1 = HVAC_results['h_r1']
    h_r2 = HVAC_results['h_r2']
    v = HVAC_results['v']
    Hum = HVAC_results['Hum']
    price = HVAC_results['price']
    Occ_r1 = HVAC_results['Occ_r1']
    Occ_r2 = HVAC_results['Occ_r2']
    
    T = range(len(Temp_r1))
    
    if axes is None:
        fig, axes = plt.subplots(4, 1, figsize=(12, 14), sharex=True)

    # Room Temperatures
    axes[0].plot(T, Temp_r1, label='Room 1 Temp', marker='o')
    axes[0].plot(T, Temp_r2, label='Room 2 Temp', marker='s')
    axes[0].axhline(18, color='gray', linestyle='--', alpha=0.5)
    axes[0].axhline(20, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_ylabel("Temperature (°C)")
    axes[0].set_title("Room Temperatures")
    axes[0].legend()
    axes[0].grid(True)
    
    # Heater consumption
    axes[1].bar(T, h_r1, width=0.4, label='Room 1 Heater', alpha=0.7)
    axes[1].bar(T, h_r2, width=0.4, label='Room 2 Heater', alpha=0.7)
    axes[1].set_ylabel("Heater Power (kW)")
    axes[1].set_title("Heater Consumption")
    axes[1].legend()
    axes[1].grid(True)
    
    # Ventilation and Humidity
    axes[2].step(T, v, where='mid', label='Ventilation ON', color='tab:blue')
    axes[2].plot(T, Hum, label='Humidity (%)', color='tab:orange', marker='o')
    axes[2].axhline(45, color='gray', linestyle='--', alpha=0.5)
    axes[2].axhline(60, color='gray', linestyle='--', alpha=0.5)
    axes[2].set_ylabel("Ventilation / Humidity")
    axes[2].set_title("Ventilation Status and Humidity")
    axes[2].legend()
    axes[2].grid(True)
    
    # Electricity price and occupancy
    axes[3].plot(T, price, label='TOU Price (€/kWh)', color='tab:red', marker='x')
    axes[3].bar(T, Occ_r1, label='Occupancy Room 1', alpha=0.5)
    axes[3].bar(T, Occ_r2, bottom=Occ_r1, label='Occupancy Room 2', alpha=0.5)
    axes[3].set_ylabel("Price / Occupancy")
    axes[3].set_xlabel("Time (hours)")
    axes[3].set_title("Electricity Price and Occupancy")
    axes[3].legend()
    axes[3].grid(True)


# General model
This is the hindsight model as specified in our report

In [23]:
def hindsight_model(price, Occ_r1, Occ_r2, fixed_data, outputFlag=0):

    T = fixed_data['num_timeslots']
    T_range = range(T)
    R = [0, 1]
    M = 70
    M_hum = 100 - fixed_data['humidity_threshold']
    model = gp.Model("Restaurant_Energy_Optimization")
    
    # Initiate all decision variables
    # Temperature
    Temp_r1 = model.addVars(T, lb=-GRB.INFINITY, name="Temp_r1")
    Temp_r2 = model.addVars(T, lb=-GRB.INFINITY, name="Temp_r2")

    # Power consumption
    h_r1 = model.addVars(T, lb=0, ub=fixed_data['heating_max_power'], name="h_r1")
    h_r2 = model.addVars(T, lb=0, ub=fixed_data['heating_max_power'], name="h_r2")
    
    # ventilation status
    v = model.addVars(T+2, vtype=GRB.BINARY, name="v")
    
    # Humidity 
    Hum = model.addVars(T, lb=0, ub=100, name="Hum")
    
    # start-up variable
    s = model.addVars(T, vtype=GRB.BINARY, name="s")
    
    # Overrule controllers
    z_low = model.addVars(R, T, vtype=GRB.BINARY, name="z_low")
    z_high = model.addVars(R, T, vtype=GRB.BINARY, name="z_high")

    # initial conditions
    model.addConstr(Temp_r1[0] == fixed_data['initial_temperature'])
    model.addConstr(Temp_r2[0] == fixed_data['initial_temperature'])
    model.addConstr(Hum[0] == fixed_data['initial_humidity'])

    model.setObjective(
        gp.quicksum(
            (fixed_data['ventilation_power'] * v[t] + h_r1[t] + h_r2[t]) * price[t]
            for t in T_range
        ), GRB.MINIMIZE
    )

    # HUmidity Dynamics
    model.addConstrs((Hum[t] == Hum[t-1] + fixed_data['humidity_occupancy_coeff'] * (Occ_r1[t-1] + Occ_r2[t-1]) -
                    fixed_data['humidity_vent_coeff'] * v[t-1] for t in range(1,T)))
    
    # Tmperature Dynamics for both rooms
    model.addConstrs((Temp_r1[t] == Temp_r1[t-1] + fixed_data['heat_exchange_coeff'] * (Temp_r2[t-1] - Temp_r1[t-1]) +
                    fixed_data['thermal_loss_coeff'] * (fixed_data['outdoor_temperature'][t-1] - Temp_r1[t-1]) +
                    fixed_data['heating_efficiency_coeff'] * h_r1[t-1] - fixed_data['heat_vent_coeff'] * v[t-1] +
                    fixed_data['heat_occupancy_coeff'] * Occ_r1[t-1] for t in range(1,T)))
    model.addConstrs((Temp_r2[t] == Temp_r2[t-1] + fixed_data['heat_exchange_coeff'] * (Temp_r1[t-1] - Temp_r2[t-1]) +
                    fixed_data['thermal_loss_coeff'] * (fixed_data['outdoor_temperature'][t-1] - Temp_r2[t-1]) +
                    fixed_data['heating_efficiency_coeff'] * h_r2[t-1] - fixed_data['heat_vent_coeff'] * v[t-1] +
                    fixed_data['heat_occupancy_coeff'] * Occ_r2[t-1] for t in range(1,T)))
    
    # Temp-min overrule controller for both rooms
    model.addConstrs((fixed_data['temp_min_comfort_threshold'] - Temp_r1[t] <= M * z_low[0, t] for t in T_range))
    model.addConstrs((fixed_data['temp_min_comfort_threshold'] - Temp_r2[t] <= M * z_low[1, t] for t in T_range))
    model.addConstrs((z_low[0, t] >= z_low[0, t-1] - 1 + ((fixed_data['temp_OK_threshold'] - Temp_r1[t]) / M) for t in range(1,T)))
    model.addConstrs((z_low[1, t] >= z_low[1, t-1] - 1 + ((fixed_data['temp_OK_threshold'] - Temp_r2[t]) / M) for t in range(1,T)))
    model.addConstrs((h_r1[t] >= z_low[0, t] * fixed_data['heating_max_power'] for t in T_range))
    model.addConstrs((h_r2[t] >= z_low[1, t] * fixed_data['heating_max_power'] for t in T_range))
    
    # Temp-max overrule controller for both rooms
    model.addConstrs((Temp_r1[t] - fixed_data['temp_max_comfort_threshold'] <= M * z_high[0, t] for t in T_range))
    model.addConstrs((Temp_r2[t] - fixed_data['temp_max_comfort_threshold'] <= M * z_high[1, t] for t in T_range))
    model.addConstrs((h_r1[t] <= (1 - z_high[0, t]) * fixed_data['heating_max_power'] for t in T_range))
    model.addConstrs((h_r2[t] <= (1 - z_high[1, t]) * fixed_data['heating_max_power'] for t in T_range))
    
    # Humidity Overrule Controller
    model.addConstrs((Hum[t] <= fixed_data['humidity_threshold'] + M_hum * v[t]) for t in T_range)
    
    # Ventilation inertia
    model.addConstr(v[0]==s[0])
    model.addConstrs((s[t] >= v[t] - v[t-1] for t in range(1,T)))
    model.addConstrs((v[t] + v[t+1] + v[t+2] >= 3 * s[t] for t in range(T)))
    

    model.setParam("OutputFlag", outputFlag)
    model.optimize()
    

    # Save and return results
    Temp_r1_vals = [Temp_r1[t].X for t in T_range]
    Temp_r2_vals = [Temp_r2[t].X for t in T_range]
    h_r1_vals = [h_r1[t].X for t in T_range]
    h_r2_vals = [h_r2[t].X for t in T_range]
    v_vals = [v[t].X for t in T_range]
    Hum_vals = [Hum[t].X for t in T_range]
    s_vals = [s[t].X for t in T_range]
    z_high_vals = [[z_high[r, t].X for t in T_range] for r in R]
    z_low_vals = [[z_low[r, t].X for t in T_range] for r in R]

    HVAC_results = {
        "Temp_r1": Temp_r1_vals,
        "Temp_r2": Temp_r2_vals,
        "h_r1": h_r1_vals,
        "h_r2": h_r2_vals,
        "v": v_vals,
        "s": s_vals,
        "z_high": z_high_vals,
        "z_low": z_low_vals,
        "Hum": Hum_vals,
        "price": price,
        "Occ_r1": Occ_r1,
        "Occ_r2": Occ_r2,
        "outdoor_temperature": fixed_data['outdoor_temperature'],
        "avg_cost": model.ObjVal
    }

    return HVAC_results

# 100 day simulation
Here we simulate our hindsight model for all 100 days. \
We output the average and plots for the days of your choice as specified in days_to_plot \
*Please note:  We changed the PlotsRestaurant.py file slightly to use it to display our resulyts (Not the structure, just variables ...)*

In [24]:
# Load all data
fixed_data = data.get_fixed_data()
price_data = pd.read_csv('PriceData.csv')
occ_r1_data = pd.read_csv('OccupancyRoom1.csv')
occ_r2_data = pd.read_csv('OccupancyRoom2.csv')

# SOlve MILP for all 100 days and store results
all_results_prof = []
all_results_ours = []
for day in range(price_data.shape[0]):
    price = price_data.iloc[day, :].tolist()
    Occ_r1 = occ_r1_data.iloc[day, :].tolist()
    Occ_r2 = occ_r2_data.iloc[day, :].tolist()
    
    result = professor_model(price, Occ_r1, Occ_r2, fixed_data)
    all_results_prof.append(result)
    results = hindsight_model(price, Occ_r1, Occ_r2, fixed_data)
    all_results_ours.append(results)






# Output average cost across all days
average_cost = np.mean([res['avg_cost'] for res in all_results_ours])
print(f"Average cost across all days ours: {average_cost:.2f}")
print(f"Average cost across all days professor: {np.mean([res['avg_cost'] for res in all_results_prof]):.2f}")

Average cost across all days ours: 111.57
Average cost across all days professor: 111.57


In [25]:
for res in range(len(all_results_ours)):
    # if costs betwen models are different, print day and costs
    if abs(all_results_ours[res]['avg_cost'] - all_results_prof[res]['avg_cost']) > 1e-2:
        print(f"Day {res}: Our cost = {all_results_ours[res]['avg_cost']:.2f}, Professor cost = {all_results_prof[res]['avg_cost']:.2f}")
        # print figure where we see them side by side in one figure (professors left, ours right)
        plot_HVAC_results(all_results_prof[res])
        plot_HVAC_results(all_results_ours[res])
        plt.tight_layout()
        plt.show()
# Output results of specified days
# days_to_plot = [0,1]
# for day in days_to_plot:
#     print(f"\nPlotting results for day {day}")
#     # PlotsRestaurant.plot_HVAC_results(all_results[day])
#     plot_HVAC_results(all_results[day]) # In case you want to use the Plotting function below
